In [1]:
!pip install tmu

In [2]:
from tmu.preprocessing.standard_binarizer.binarizer import StandardBinarizer
from tmu.models.regression.vanilla_regressor import TMRegressor
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
import logging
import argparse
from tmu.tools import BenchmarkTimer

_LOGGER = logging.getLogger(__name__)

# Initialize StandardBinarizer and transform a sample of X for global access
california_housing = datasets.fetch_california_housing()
X_sample = california_housing.data

b_global = StandardBinarizer(max_bits_per_feature=10)
X_transformed_global = b_global.fit_transform(X_sample)

print(f"X_transformed_global.shape[1]: {X_transformed_global.shape[1]}")

ERROR:tmu.clause_bank.clause_bank_cuda:No module named 'pycuda'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank_cuda.py", line 41, in <module>
    from pycuda._driver import Device, Context
ModuleNotFoundError: No module named 'pycuda'


X_transformed_global.shape[1]: 80


In [3]:
def metrics(args):
    return dict(
        mae=[],
        train_time=[],
        test_time=[]
    )


In [4]:
def main(args):
    tm_results = np.empty(0)
    experiment_results = metrics(args)

    california_housing = datasets.fetch_california_housing()
    X = california_housing.data
    Y = california_housing.target

    # Use the globally defined binarizer
    X_transformed = b_global.fit_transform(X)

    print(f"Shape of X_transformed: {X_transformed.shape}")
    print(f"Number of clauses (args.num_clauses): {args.num_clauses}")
    print(f"2 * X_transformed.shape[1]: {2 * X_transformed.shape[1]}")
    if args.num_clauses == 2 * X_transformed.shape[1]:
        print("Clause size is 2 times the number of features in X_transformed.")
    else:
        print("Clause size is NOT 2 times the number of features in X_transformed.")

    tm = TMRegressor(
        args.num_clauses,
        args.T,
        args.s,
        platform=args.platform,
        weighted_clauses=args.weighted_clauses
    )

    tm_results = np.empty(0)

    _LOGGER.info(f"Running MAE with {TMRegressor} for {args.num_runs}")
    for run in range(args.num_runs):
        X_train, X_test, Y_train, Y_test = train_test_split(X_transformed, Y)

        for epoch in range(args.epochs):
            benchmark1 = BenchmarkTimer(logger=_LOGGER, text="Training Time")
            with benchmark1:
                tm.fit(X_train, Y_train)
            experiment_results["train_time"].append(benchmark1.elapsed())

            benchmark2 = BenchmarkTimer(logger=_LOGGER, text="Testing Time")
            with benchmark2:
               # tm_results = np.append(tm_results, np.sqrt(((tm.predict(X_test) - Y_test) ** 2).mean()))
                 tm_results = np.append(tm_results, np.abs(tm.predict(X_test) - Y_test).mean())
            Y_pred = tm.predict(X_test)

            # Save the final test set
            final_X_test = X_test.copy()
            final_Y_test = Y_test.copy()

            #print("X_test shape:", X_test.shape)
            #print("Y_test shape:", Y_test.shape)




            experiment_results["test_time"].append(benchmark2.elapsed())
            experiment_results["mae"].append(tm_results.mean())



            _LOGGER.info(
                f"#{run + 1} - Epoch: {epoch}: MAE: {tm_results.mean()} +/- {1.96 * tm_results.std() / np.sqrt(run + 1)} ({benchmark1.elapsed()})")


        # print("T =", tm.T)
        # print("MAX_Y =", tm.max_y)
        # print("MIN_Y =", tm.min_y)


    # the 3 lines given below are clause and weight extraction methods
    weights = tm.weight_bank.get_weights()
    print(dir(tm.clause_bank))
    clauses = tm.clause_bank.get_literals().astype(np.uint8)

    print("=== CLAUSE BANK INFO ===")
    print(type(clauses))
    print("Shape:", np.shape(clauses))
    print("ndim:", np.ndim(clauses))
    print("dtype:", clauses.dtype)

    print("actions type:", type(tm.clause_bank.actions))

    try:
        print("actions shape:", tm.clause_bank.actions.shape)
    except Exception as e:
        print("shape error:", e)

    print("actions object:")
    print(tm.clause_bank.actions)

    try:
        lits = tm.clause_bank.get_literals()

        print("lits type:", type(lits))
        print("lits shape:", np.shape(lits))

        print(lits[:10])
    except Exception as e:
        print("get_literals error:", e)

    if len(clauses) > 0:
       print("First element type:", type(clauses[0]))
       print("First element:", clauses[0])


    #words_per_clause = len(clauses) // args.num_clauses



    #print("Clause bank shape:", clauses.shape)

    #Extracting all the files

    with open("clauses.mem", "w") as f:

        for clause in clauses:
            f.write("".join(map(str, clause)) + "\n")

    with open("weights.mem", "w") as f:
       for w in weights:
           f.write(
               f"{bin(int(w) & 0xFFFFFFFF)[2:].zfill(32)}\n"
           )
    with open("Xtest_160.mem", "w") as f:
       for row in X_test.astype(np.uint8):
           pos = row
           neg = 1 - row

           expanded = np.concatenate([pos, neg])

           f.write("".join(map(str, expanded)) + "\n")

    # with open("Ytest.txt", "w") as f:
    #    for y in final_Y_test:
    #        f.write(f"{y}\n")
    with open("Ytest.mem", "w") as f:
       for y in final_Y_test[:100]:
           fixed = int(round(y * 100000))
           f.write(f"{fixed:032b}\n")

   # print("Number of test samples =", len(X_test))
    print("MEM FILES SAVED SUCCESSFULLY IN BINARY FORMAT")

    final_mae = tm_results.mean()

    #from sklearn.metrics import r2_score

    for i in range(10):
           print(i, int(Y_pred[i]*100000), int(Y_test[i]*100000))

    print("\nFINAL RESULTS")
    print("FINAL MAE:", final_mae)
    #print("FINAL R2 Score:", r2_score(Y_test, Y_pred))

    return experiment_results

In [5]:
!sed -i 's/np.uint32(~0)/np.uint32(0xFFFFFFFF)/g' /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py

In [6]:
!grep -n "uint32" /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py | head

60:        self.clause_output = np.empty(self.number_of_clauses, dtype=np.uint32, order="c")
61:        self.clause_output_batch = np.empty(self.number_of_clauses * batch_size, dtype=np.uint32, order="c")
62:        self.clause_and_target = np.zeros(self.number_of_clauses * self.number_of_ta_chunks, dtype=np.uint32, order="c")
63:        self.clause_output_patchwise = np.empty(self.number_of_clauses * self.number_of_patches, dtype=np.uint32, order="c")
64:        self.feedback_to_ta = np.empty(self.number_of_ta_chunks, dtype=np.uint32, order="c")
65:        self.output_one_patches = np.empty(self.number_of_patches, dtype=np.uint32, order="c")
66:        self.literal_clause_count = np.empty(self.number_of_literals, dtype=np.uint32, order="c")
69:        self.type_ia_feedback_counter = np.zeros(self.number_of_clauses, dtype=np.uint32, order="c")
74:            dtype=np.uint32,
79:            dtype=np.uint32,


In [7]:
!sed -n '130,140p' /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py

        self.clause_bank = np.empty(
            shape=(self.number_of_clauses, self.number_of_ta_chunks, self.number_of_state_bits_ta),
            dtype=np.uint32,
            order="c"
        )

        self.clause_bank[:, :, 0: self.number_of_state_bits_ta - 1] = np.uint32(0xFFFFFFFF)
        self.clause_bank[:, :, self.number_of_state_bits_ta - 1] = 0
        self.clause_bank = np.ascontiguousarray(self.clause_bank.reshape(
            (self.number_of_clauses * self.number_of_ta_chunks * self.number_of_state_bits_ta)))



In [8]:
!sed -i 's/np.uint32(0xFFFFFFFF)/np.uint32(2**32 - 1)/g' /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py

In [9]:
!sed -i 's/np.uint32(~0)/np.uint32(2**32 - 1)/g' /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py

In [10]:
!sed -i 's/np.uint32(2\*\*32 - 1)/np.array(0xFFFFFFFF, dtype=np.uint32)/g' /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py

In [5]:
def default_args(**kwargs):
    parser = argparse.ArgumentParser()
    parser.add_argument("--num_clauses", default=1000, type=int)
    parser.add_argument("--T", default=500 * 10, type=int)
    parser.add_argument("--s", default=2.75, type=float)
    parser.add_argument("--platform", default="CPU", type=str)
    parser.add_argument("--weighted_clauses", default=True, type=bool)
    parser.add_argument("--epochs", default=30, type=int)
    parser.add_argument("--num-runs", default=25, type=int)
    args, unknown = parser.parse_known_args()
    for key, value in kwargs.items():
        if key in args.__dict__:
            setattr(args, key, value)
    return args


if __name__ == "__main__":
    results = main(default_args())
    _LOGGER.info(results)

Shape of X_transformed: (20640, 80)
Number of clauses (args.num_clauses): 1000
2 * X_transformed.shape[1]: 160
Clause size is NOT 2 times the number of features in X_transformed.


KeyboardInterrupt: 